*[cell 0]*

# LVL_0 Report

Descriptive tables on one stamped LVL_0 handoff, pinned as `LVL0_HANDOFF` in `[cell 1]`. The report reads the handoff,
its run log (`lv0_out_<ts>_parameters.json`: the intake gates and the drift models LVL_0 fitted) and the group-level
decay fits the LVL0 notebook exports beside the handoff (`lv0_out_<ts>_decay_fits.csv`). It fits nothing. Set
`PROJECT_ROOT` (README, "Paths") and the pin in `[cell 1]`.

| Table | Content | Export |
|---|---|---|
| 0.2 | Session duration (summed response times) by condition and session | `session_duration` |
| 0.2b | Session duration: S vs S per condition (consecutive sessions; Wilcoxon signed-rank) | `session_duration_tests` |
| 0.4 | Responses outside the RT limits | `rt_limits` |
| 0.5 | Drift models: the model per span and its fit per scale | `drift_models` |
| 0.6 | Drift-model selection: mean R² of every candidate per span | `model_selection` |

Each table is written to `LVL_OUTS` as `<handoff stem>_report_<export>.csv`; the final report prints them. The
decay-curve figures are drawn by `make_replication_figures.ipynb` from the LVL0 notebook's exports; the sample
composition is `demographic_report.ipynb`'s.

In [ ]:
# [cell 1]
# ===== INPUTS: set these before running =====
LVL0_HANDOFF = None   # the LVL0 output in LVL_OUTS, e.g. "lv0_out_20261005_101500.csv"
PROJECT_ROOT = "SET_ME"   # the absolute project folder holding code/, dset/ and dervatives/ (README, "Paths"; EMST /home/sjsuss626/EMST_files/sjs_dissertation)

_SET_ME = {"PROJECT_ROOT": (PROJECT_ROOT, 'the absolute project folder holding code/, dset/ and dervatives/; the '
                                          'README\'s one-line sed command sets it (README, "Paths"); EMST '
                                          '/home/sjsuss626/EMST_files/sjs_dissertation')}   # name: (value, what to set)
_unset = [k for k, (v, _) in _SET_ME.items() if isinstance(v, str) and v == "SET_ME"]
if _unset:
    raise SystemExit(f'settings check (LVL0_report.ipynb): {len(_unset)} setting(s) still hold "SET_ME". Set each, then run '
                     'again (README, "Settings"):\n' + "\n".join(
                         f"  - {k}  (LVL0_report [cell 1])\n      {_SET_ME[k][1]}" for k in _unset))

# =============================================================================
# IMPORTS, INPUT, PATHS
# =============================================================================
import json, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

# ---- locations: everything is inside PROJECT_ROOT ----------------
_OUT = Path(PROJECT_ROOT) / "dervatives" / "practical_application_generic"
LVL_OUTS = _OUT / "lvl_outs"   # the handoff, its exports; this report's tables

if LVL0_HANDOFF is None:
    raise SystemExit("LVL0_HANDOFF is not set (top of this cell). Name one LVL_0 handoff; "
                     f"available in {LVL_OUTS}: {sorted(p.name for p in LVL_OUTS.glob('lv0_out_*.csv'))}")
LV0_PATH = LVL_OUTS / LVL0_HANDOFF
_m0 = re.match(r"lv0_out_(\d{8}_\d{6})", LV0_PATH.name)
if not _m0 or LV0_PATH.suffix != ".csv" or not LV0_PATH.is_file():
    raise SystemExit(f"LVL0_HANDOFF = {LVL0_HANDOFF!r} is not an existing LVL_0 handoff "
                     "(lv0_out_<YYYYmmdd_HHMMSS>[...].csv)")
# The handoff's LVL_0 run log (configuration, drift models and their fits): the one file of the same stamp beside it.
_recs = sorted(LV0_PATH.parent.glob(f"lv0_out_{_m0.group(1)}*_parameters.json"))
if len(_recs) != 1:
    raise SystemExit(f"expected exactly one LVL_0 run log lv0_out_{_m0.group(1)}*_parameters.json beside "
                     f"{LV0_PATH.name}, found {[p.name for p in _recs]}")
LVL0_RUN_LOG = json.load(open(_recs[0]))
LVL0_PARAMS = LVL0_RUN_LOG["parameters"]
# The group-level decay fits, exported by the LVL0 notebook beside the handoff.
DECAY_FITS_PATH = LV0_PATH.with_name(f"{LV0_PATH.stem}_decay_fits.csv")
if not DECAY_FITS_PATH.is_file():
    raise SystemExit(f"{DECAY_FITS_PATH.name} is missing: the LVL0 notebook's export cell writes it beside the handoff")
OUT_STEM = LVL_OUTS / f"{LV0_PATH.stem}_report"

PR_STIM_DROP  = LVL0_PARAMS["pr_stim_drop"]     # first N stimuli of each session (practice)
MIN_RT_THRESH = LVL0_PARAMS["min_rt_thresh"]    # seconds
MAX_RT_THRESH = LVL0_PARAMS["max_rt_thresh"]    # seconds
REPORT_TABLES = {}


def export(key, frame, title):
    """Write one table to LVL_OUTS as <handoff stem>_report_<key>.csv and show it."""
    REPORT_TABLES[key] = frame
    frame.to_csv(f"{OUT_STEM}_{key}.csv", index=False)
    print(title)
    display(frame)


def bh(p):
    """Benjamini-Hochberg q-values; NaN stays NaN."""
    p = np.asarray(p, float); q = np.full(len(p), np.nan); ok = np.isfinite(p)
    if ok.any():
        q[ok] = multipletests(p[ok], method="fdr_bh")[1]
    return q


print(f"LVL_0 handoff : {LV0_PATH}")
print(f"LVL_0 run log : {_recs[0].name}  (pr_stim_drop={PR_STIM_DROP}, min_rt_thresh={MIN_RT_THRESH}, "
      f"max_rt_thresh={MAX_RT_THRESH})")
print(f"decay fits    : {DECAY_FITS_PATH.name}")
print(f"outputs       : {OUT_STEM}_*.csv")

# Interpreter and package versions of this run.
import sys as _vsys, platform as _vplatform
from importlib import metadata as _vmeta


def _pkg_version(p):
    try:
        return _vmeta.version(p)
    except _vmeta.PackageNotFoundError:
        return "not installed"


print(f"interpreter: {_vsys.executable} (Python {_vplatform.python_version()})")
print("packages: " + ", ".join(f"{p} {_pkg_version(p)}" for p in (
    "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "hmmlearn", "ruptures", "joblib", "matplotlib")))

In [ ]:
# [cell 2]
# =============================================================================
# LOAD
# =============================================================================
# Every response of the analysed sample (the LVL_0 handoff keeps all of them; the practice stimuli and the RT limits
# are applied only where a table says so).
lv0 = pd.read_csv(LV0_PATH, usecols=["id", "condition", "session", "stim_order", "rt_sec"], low_memory=False)
print(f"LVL_0 handoff: {lv0.id.nunique()} participants, {len(lv0.drop_duplicates(['id', 'session']))} sessions, "
      f"{len(lv0):,} responses")
DECAY_FITS = pd.read_csv(DECAY_FITS_PATH)
DRIFT_FITS = pd.DataFrame(LVL0_RUN_LOG["statistical_results"]["drift_fit_parameters"])
SPAN = {lab: ("Overall" if lab.startswith("Overall") else "S" + lab.split()[-1])
        for lab in DRIFT_FITS.session_label.unique()}
# the decay-fit export's selected model per span is the model LVL_0 fitted (its run log)
_sel = DECAY_FITS[DECAY_FITS["selected"]].set_index("span")["model"].to_dict()
_run = DRIFT_FITS.assign(span=DRIFT_FITS.session_label.map(SPAN)).groupby("span").model_name.agg(set).to_dict()
if {s: {m} for s, m in _sel.items()} != _run:
    raise SystemExit(f"the decay-fit export's models {_sel} differ from the LVL_0 run log's {_run}: "
                     "re-run the LVL0 notebook's export cell on this handoff")

In [ ]:
# [cell 3]
# =============================================================================
# TABLES 0.2 / 0.2b: SESSION DURATION
# =============================================================================
# Duration = the sum of the raw response times of every response in the session (the scales of every stimulus, before
# the RT limits), in minutes; the time between responses is not recorded. Consecutive sessions S(a) vs S(b) among the
# participants with both (Wilcoxon signed-rank): per condition and, with two or more conditions, over all
# participants; q = BH over these tests. With one session per participant there is no test.
dur = lv0.groupby(["id", "condition", "session"]).rt_sec.sum().div(60).rename("minutes").reset_index()
_w = dur.pivot_table(index=["id", "condition"], columns="session", values="minutes").reset_index()
_pm = dur.groupby(["id", "condition"]).minutes.mean().reset_index()
_conds, _sess = sorted(dur.condition.unique()), sorted(dur.session.unique())
_tests = []
for lab, g in [(cd, _w[_w.condition == cd]) for cd in _conds] + ([("all participants", _w)] if len(_conds) > 1 else []):
    for a, b in zip(_sess[:-1], _sess[1:]):
        g2 = g.dropna(subset=[a, b])
        if not len(g2):
            continue
        try:
            _t = stats.wilcoxon(g2[a], g2[b])
            _stat, _p = _t.statistic, _t.pvalue
        except ValueError:                       # e.g. every difference zero
            _stat, _p = np.nan, np.nan
        _tests.append({"Comparison": f"S{a} vs S{b}, {lab} (Wilcoxon signed-rank)", "n": len(g2), "Statistic": _stat,
                       "p": _p})
_qd = bh([r["p"] for r in _tests])
_med = lambda x: f"{x.median():.1f} [{x.quantile(.25):.1f}, {x.quantile(.75):.1f}]"
_msd = lambda x: f"{x.mean():.1f} ({x.std():.1f})"
rows = [{"Group": f"{cd}, S{s}", "Sessions": len(g), "Duration, min: Mdn [Q1, Q3]": _med(g.minutes),
         "Duration, min: M (SD)": _msd(g.minutes)}
        for cd in _conds for s in _sess for g in [dur[(dur.condition == cd) & (dur.session == s)]] if len(g)]
rows += [{"Group": f"{cd} (participant mean of sessions)", "Sessions": int((dur.condition == cd).sum()),
          "Duration, min: Mdn [Q1, Q3]": _med(_pm.minutes[_pm.condition == cd]),
          "Duration, min: M (SD)": _msd(_pm.minutes[_pm.condition == cd])} for cd in _conds]
rows += [{"Group": "All sessions", "Sessions": len(dur), "Duration, min: Mdn [Q1, Q3]": _med(dur.minutes),
          "Duration, min: M (SD)": _msd(dur.minutes)}]
export("session_duration", pd.DataFrame(rows),
       "Table 0.2. Session duration (summed response times) by condition and session")
export("session_duration_tests", pd.DataFrame([{**r, "q": q} for r, q in zip(_tests, _qd)],
                                              columns=["Comparison", "n", "Statistic", "p", "q"]),
       "Table 0.2b. Session duration: S vs S per condition")

In [ ]:
# [cell 4]
# =============================================================================
# TABLE 0.4: RESPONSES OUTSIDE THE RT LIMITS
# =============================================================================
# The RT limits set a response's RT to missing at LVL_1 (every response); the practice stimuli are left out of the
# LVL_0 drift fits only.
_pr, _hi, _lo = int(PR_STIM_DROP), float(MAX_RT_THRESH), float(MIN_RT_THRESH)
rows = []
for lab, d_ in (("all responses", lv0), (f"excluding the first {_pr} (practice) stimuli of each session",
                                         lv0[lv0.stim_order > _pr])):
    rows.append({"Responses": lab, "n": len(d_),
                 f"RT > {_hi:g} s, n (%)": f"{int((d_.rt_sec > _hi).sum()):,} ({100 * (d_.rt_sec > _hi).mean():.2f})",
                 f"RT < {_lo:g} s, n (%)": f"{int((d_.rt_sec < _lo).sum()):,} ({100 * (d_.rt_sec < _lo).mean():.2f})"})
export("rt_limits", pd.DataFrame(rows), "Table 0.4. Preprocessing: RTs outside the limits")

In [ ]:
# [cell 5]
# =============================================================================
# TABLES 0.5 / 0.6: DRIFT MODELS
# =============================================================================
# Table 0.5: per span, the model LVL_0 fitted and its R^2 per scale (LVL_0's fit to the median RT in seconds, from the
# run log), with the R^2 of the same model fitted to the median log RT (the LVL0 notebook's decay-fit export: the
# curves of the decay-curve figure). Table 0.6: the mean R^2 over the scales of every candidate, per span; the highest
# is the model used.
rows = []
for r in DRIFT_FITS.itertuples():
    sp = SPAN[r.session_label]
    lg = DECAY_FITS[(DECAY_FITS["table"] == "figure fit") & (DECAY_FITS["transform"] == "log")
                    & (DECAY_FITS["span"] == sp) & (DECAY_FITS["scale"] == r.scale)]
    rows.append({"Span": r.session_label, "Scale": r.scale, "Model": r.model_name,
                 "R² (LVL_0 fit, median RT in s)": r.r_squared,
                 "R² (log-scale fit)": lg["r_squared"].iloc[0] if len(lg) else np.nan})
export("drift_models", pd.DataFrame(rows), "Table 0.5. Drift models: the selected model per span and its fit per scale")
_ms = DECAY_FITS[DECAY_FITS["table"] == "model selection"]
export("model_selection", _ms.pivot_table(index="model", columns="span", values="r_squared").reset_index()
       .rename(columns={"model": "Model"}), "Table 0.6. Drift-model selection: mean R² of every candidate, per span")